# Clase 10 - MCP con `stdio` y Streamable HTTP en Colab/local

<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Google Colab"/></a>

> Este laboratorio está diseñado para ejecutarse **por defecto en Google Colab**. También funciona en un notebook local con Python/Jupyter.

**Objetivos.** Al terminar deberías poder:

- Preparar un entorno MCP portable para Colab y ejecución local.
- Crear una base SQLite pequeña de ejemplo.
- Implementar un servidor MCP con `FastMCP`.
- Exponer `resources`, `tools` y `prompts`.
- Probar el servidor con cliente MCP por `stdio`.
- Levantar el mismo servidor con transporte `streamable-http` en `localhost`.
- Comparar cuándo conviene `stdio` y cuándo conviene Streamable HTTP.
- Revisar seguridad local: confirmación humana, logs, puertos y separación entre datos y acciones.

El flujo completo ocurre dentro del notebook. No requiere SSH, clientes de escritorio ni APIs externas.


## Arquitectura

El mismo servidor MCP se probará con dos métodos de transporte vistos en clases.

### Parte A: `stdio`

```text
Google Colab o Jupyter local
          |
          | cliente MCP en Python
          | stdio + JSON-RPC
          v
Servidor MCP curso-mcp como subproceso
          |
          | sqlite3 local
          v
Base curso_mcp.db
```

### Parte B: Streamable HTTP

```text
Google Colab o Jupyter local
          |
          | cliente MCP en Python
          | HTTP local: http://127.0.0.1:PUERTO/mcp
          v
Servidor MCP curso-mcp escuchando en localhost
          |
          | sqlite3 local
          v
Base curso_mcp.db
```

`stdio` es ideal para ejecución local simple. Streamable HTTP es útil cuando el servidor debe aceptar conexiones vía red, manejar más de un cliente o integrarse como servicio.


## Paso 0: compatibilidad

| Entorno | `stdio` | Streamable HTTP local |
|---|---:|---:|
| Google Colab | sí | sí, dentro del runtime |
| Jupyter local en Linux/macOS/Windows | sí | sí, en `127.0.0.1` |

En este laboratorio Streamable HTTP se ejecuta solo en `localhost`. Para exponerlo fuera del entorno tendrías que agregar autenticación, TLS, control de origen, rate limiting y políticas de permisos.


In [66]:
# Instalación recomendada para Colab o entorno local limpio.
%pip install -q -U "mcp[cli]" pandas==2.2.2 pydantic nest_asyncio

In [67]:
import json
import os
import platform
import socket
import sqlite3
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
print("Ejecutable Python:", sys.executable)

Python: 3.12.13
Sistema: Linux-6.6.122+-x86_64-with-glibc2.35
Ejecutable Python: /usr/bin/python3


## Paso 1: crear carpeta de trabajo

In [75]:
if Path("/content").exists():
    LAB_DIR = Path("/content/mcp_lab")
    ENTORNO = "Google Colab"
else:
    LAB_DIR = Path.cwd() / "mcp_lab"
    ENTORNO = "Jupyter/local"

LAB_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = LAB_DIR / "curso_mcp.db"
SERVER_PATH = LAB_DIR / "servidor_curso_mcp.py"
LOG_PATH = Path("/content/mcp_server_error.log")

print("Entorno detectado:", ENTORNO)
print("LAB_DIR:", LAB_DIR)
print("DB_PATH:", DB_PATH)
print("SERVER_PATH:", SERVER_PATH)


Entorno detectado: Google Colab
LAB_DIR: /content/mcp_lab
DB_PATH: /content/mcp_lab/curso_mcp.db
SERVER_PATH: /content/mcp_lab/servidor_curso_mcp.py


## Paso 2: crear base SQLite demo

La base simula datos del curso. Luego el servidor MCP la expondrá como contexto y herramientas.

In [69]:
def inicializar_sqlite(db_path: Path):
    if db_path.exists():
        db_path.unlink()
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("""
    CREATE TABLE clases (
        numero INTEGER PRIMARY KEY,
        titulo TEXT NOT NULL,
        tema TEXT NOT NULL,
        resumen TEXT NOT NULL,
        duracion_min INTEGER NOT NULL
    )
    """)
    cur.execute("""
    CREATE TABLE tareas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        clase_numero INTEGER NOT NULL,
        titulo TEXT NOT NULL,
        prioridad TEXT NOT NULL,
        estado TEXT NOT NULL,
        FOREIGN KEY(clase_numero) REFERENCES clases(numero)
    )
    """)
    clases = [
        (1, "Cuantización, PEFT y despliegue", "LLMs eficientes", "Reduce memoria, adapta modelos con LoRA/QLoRA y despliega de forma responsable.", 120),
        (2, "Aprendizaje por contexto", "Prompting avanzado", "Diseña prompts zero-shot, few-shot, CoT, autoconsistencia y salida estructurada.", 120),
        (3, "Agentes", "Agentes con herramientas", "Construye agentes que perciben, planifican, actúan y registran memoria.", 120),
        (4, "Model Context Protocol", "Integración estándar", "Conecta hosts LLM con tools, resources y prompts mediante MCP.", 120),
    ]
    cur.executemany("INSERT INTO clases VALUES (?, ?, ?, ?, ?)", clases)
    tareas = [
        (1, "Probar LoRA con modelo pequeño", "media", "pendiente"),
        (2, "Comparar zero-shot vs few-shot", "alta", "completada"),
        (3, "Conectar agente a Firestore simulado", "alta", "pendiente"),
        (4, "Crear servidor MCP por stdio", "critica", "pendiente"),
    ]
    cur.executemany("INSERT INTO tareas (clase_numero, titulo, prioridad, estado) VALUES (?, ?, ?, ?)", tareas)
    con.commit()
    con.close()

inicializar_sqlite(DB_PATH)
print("Base creada:", DB_PATH)

Base creada: /content/mcp_lab/curso_mcp.db


In [70]:
with sqlite3.connect(DB_PATH) as con:
    display(pd.read_sql_query("SELECT * FROM clases", con))
    display(pd.read_sql_query("SELECT * FROM tareas", con))

,numero,titulo,tema,resumen,duracion_min
0,1,"Cuantización, PEFT y despliegue",LLMs eficientes,"Reduce memoria, adapta modelos con LoRA/QLoRA ...",120
1,2,Aprendizaje por contexto,Prompting avanzado,"Diseña prompts zero-shot, few-shot, CoT, autoc...",120
2,3,Agentes,Agentes con herramientas,"Construye agentes que perciben, planifican, ac...",120
3,4,Model Context Protocol,Integración estándar,"Conecta hosts LLM con tools, resources y promp...",120


,id,clase_numero,titulo,prioridad,estado
0,1,1,Probar LoRA con modelo pequeño,media,pendiente
1,2,2,Comparar zero-shot vs few-shot,alta,completada
2,3,3,Conectar agente a Firestore simulado,alta,pendiente
3,4,4,Crear servidor MCP por stdio,critica,pendiente


## Paso 3: contrato del servidor MCP

| Tipo | Nombre/URI | Uso |
|---|---|---|
| Resource | `curso://resumen` | Resumen completo del curso |
| Resource | `clase://{numero}` | Detalle de una clase |
| Resource | `schema://sqlite` | Esquema de la base |
| Tool | `buscar_clases` | Buscar por texto |
| Tool | `obtener_clase` | Obtener clase estructurada |
| Tool | `listar_tareas` | Listar tareas filtradas |
| Tool | `crear_tarea` | Crear tarea con confirmación |
| Tool | `marcar_tarea` | Cambiar estado con confirmación |
| Tool | `resumen_progreso` | Métricas simples |
| Prompt | `preparar_estudio` | Guía de estudio |
| Prompt | `generar_quiz` | Plantilla de quiz |

## Paso 4: escribir servidor MCP

El servidor usa `FastMCP`. El mismo archivo puede arrancar en dos modos:

- `stdio`: comunicación por entrada/salida estándar, sin abrir puertos.
- `streamable-http`: servidor HTTP local en `/mcp`.

En modo `stdio`, `stdout` queda reservado para JSON-RPC; los logs deben ir a `stderr`. En modo HTTP, el servidor escucha en `127.0.0.1` para mantenerlo local al runtime de Colab/Jupyter.


In [94]:
server_code = '\nimport argparse\nimport json\nimport os\nimport sqlite3\nimport sys\nfrom pathlib import Path\nfrom typing import Any, Optional\n\nfrom mcp.server import MCPServer\n\nDB_PATH = Path(os.environ.get("CURSO_MCP_DB", "curso_mcp.db")).resolve()\nmcp = MCPServer("curso-mcp")\n\ndef conectar():\n    if not DB_PATH.exists():\n        raise ValueError(f"No existe la base SQLite: {DB_PATH}")\n    con = sqlite3.connect(DB_PATH)\n    con.row_factory = sqlite3.Row\n    return con\n\ndef filas_dict(cursor):\n    return [dict(row) for row in cursor.fetchall()]\n\ndef validar_prioridad(prioridad: str) -> str:\n    prioridad = prioridad.lower().strip()\n    validas = {"baja", "media", "alta", "critica"}\n    if prioridad not in validas:\n        raise ValueError(f"Prioridad inválida: {prioridad}. Usa una de: {sorted(validas)}")\n    return prioridad\n\ndef validar_estado(estado: str) -> str:\n    estado = estado.lower().strip()\n    validos = {"pendiente", "en_progreso", "completada"}\n    if estado not in validos:\n        raise ValueError(f"Estado inválido: {estado}. Usa uno de: {sorted(validos)}")\n    return estado\n\n@mcp.resource("curso://resumen", mime_type="application/json")\ndef recurso_resumen_curso() -> str:\n    """Devuelve un resumen JSON de clases y tareas del curso."""\n    with conectar() as con:\n        clases = filas_dict(con.execute("SELECT * FROM clases ORDER BY numero"))\n        tareas = filas_dict(con.execute("SELECT * FROM tareas ORDER BY id"))\n    return json.dumps({"clases": clases, "tareas": tareas}, ensure_ascii=False, indent=2)\n\n@mcp.resource("clase://{numero}", mime_type="application/json")\ndef recurso_clase(numero: str) -> str:\n    """Devuelve el detalle JSON de una clase por número."""\n    try:\n        numero_int = int(numero)\n    except ValueError:\n        raise ValueError("El número de clase debe ser entero.")\n    with conectar() as con:\n        row = con.execute("SELECT * FROM clases WHERE numero = ?", (numero_int,)).fetchone()\n        if not row:\n            raise ValueError(f"No existe la clase {numero_int}.")\n        tareas = filas_dict(con.execute("SELECT * FROM tareas WHERE clase_numero = ? ORDER BY id", (numero_int,)))\n    return json.dumps({"clase": dict(row), "tareas": tareas}, ensure_ascii=False, indent=2)\n\n@mcp.resource("schema://sqlite", mime_type="text/plain")\ndef recurso_schema() -> str:\n    """Devuelve el esquema SQL disponible para el servidor."""\n    with conectar() as con:\n        rows = con.execute("SELECT name, sql FROM sqlite_master WHERE type=\'table\' ORDER BY name").fetchall()\n    return "\\n\\n".join([f"-- {r[\'name\']}\\n{r[\'sql\']}" for r in rows])\n\n@mcp.tool()\ndef buscar_clases(termino: str) -> list[dict[str, Any]]:\n    """Busca clases por título, tema o resumen."""\n    termino_like = f"%{termino.lower().strip()}%"\n    with conectar() as con:\n        cur = con.execute("""\n            SELECT * FROM clases\n            WHERE lower(titulo) LIKE ? OR lower(tema) LIKE ? OR lower(resumen) LIKE ?\n            ORDER BY numero\n        """, (termino_like, termino_like, termino_like))\n        return filas_dict(cur)\n\n@mcp.tool()\ndef obtener_clase(numero: int) -> dict[str, Any]:\n    """Obtiene una clase y sus tareas asociadas."""\n    with conectar() as con:\n        row = con.execute("SELECT * FROM clases WHERE numero = ?", (numero,)).fetchone()\n        if not row:\n            raise ValueError(f"No existe la clase {numero}.")\n        tareas = filas_dict(con.execute("SELECT * FROM tareas WHERE clase_numero = ? ORDER BY id", (numero,)))\n    return {"clase": dict(row), "tareas": tareas}\n\n@mcp.tool()\ndef listar_tareas(estado: Optional[str] = None, prioridad: Optional[str] = None) -> list[dict[str, Any]]:\n    """Lista tareas opcionalmente filtradas por estado y prioridad."""\n    filtros = []\n    params: list[Any] = []\n    if estado:\n        filtros.append("estado = ?")\n        params.append(validar_estado(estado))\n    if prioridad:\n        filtros.append("prioridad = ?")\n        params.append(validar_prioridad(prioridad))\n    where = " WHERE " + " AND ".join(filtros) if filtros else ""\n    with conectar() as con:\n        cur = con.execute(f"SELECT * FROM tareas{where} ORDER BY id", params)\n        return filas_dict(cur)\n\n@mcp.tool()\ndef crear_tarea(clase_numero: int, titulo: str, prioridad: str = "media", confirmado: bool = False) -> dict[str, Any]:\n    """Crea una tarea. Requiere confirmado=True porque escribe en la base."""\n    if not confirmado:\n        raise ValueError("Crear tareas modifica la base. Reintenta con confirmado=True si el usuario lo aprueba.")\n    prioridad = validar_prioridad(prioridad)\n    titulo = titulo.strip()\n    if not titulo:\n        raise ValueError("El título no puede estar vacío.")\n    with conectar() as con:\n        clase = con.execute("SELECT numero FROM clases WHERE numero = ?", (clase_numero,)).fetchone()\n        if not clase:\n            raise ValueError(f"No existe la clase {clase_numero}.")\n        cur = con.execute("INSERT INTO tareas (clase_numero, titulo, prioridad, estado) VALUES (?, ?, ?, \'pendiente\')", (clase_numero, titulo, prioridad))\n        con.commit()\n        row = con.execute("SELECT * FROM tareas WHERE id = ?", (cur.lastrowid,)).fetchone()\n        return dict(row)\n\n@mcp.tool()\ndef marcar_tarea(tarea_id: int, estado: str, confirmado: bool = False) -> dict[str, Any]:\n    """Cambia el estado de una tarea. Requiere confirmado=True."""\n    if not confirmado:\n        raise ValueError("Cambiar estado modifica la base. Reintenta con confirmado=True si el usuario lo aprueba.")\n    estado = validar_estado(estado)\n    with conectar() as con:\n        row = con.execute("SELECT * FROM tareas WHERE id = ?", (tarea_id,)).fetchone()\n        if not row:\n            raise ValueError(f"No existe la tarea {tarea_id}.")\n        con.execute("UPDATE tareas SET estado = ? WHERE id = ?", (estado, tarea_id))\n        con.commit()\n        row = con.execute("SELECT * FROM tareas WHERE id = ?", (tarea_id,)).fetchone()\n        return dict(row)\n\n@mcp.tool()\ndef resumen_progreso() -> dict[str, Any]:\n    """Devuelve métricas de avance del curso y tareas."""\n    with conectar() as con:\n        total_clases = con.execute("SELECT COUNT(*) AS n FROM clases").fetchone()["n"]\n        total_tareas = con.execute("SELECT COUNT(*) AS n FROM tareas").fetchone()["n"]\n        por_estado = filas_dict(con.execute("SELECT estado, COUNT(*) AS cantidad FROM tareas GROUP BY estado ORDER BY estado"))\n        criticas = filas_dict(con.execute("SELECT * FROM tareas WHERE prioridad=\'critica\' AND estado!=\'completada\' ORDER BY id"))\n    return {"total_clases": total_clases, "total_tareas": total_tareas, "tareas_por_estado": por_estado, "tareas_criticas_pendientes": criticas}\n\n@mcp.prompt(title="Preparar estudio")\ndef preparar_estudio(numero_clase: int) -> str:\n    """Genera una plantilla para estudiar una clase."""\n    return f"""Prepara una guía de estudio para la clase {numero_clase}.\nUsa el recurso clase://{numero_clase} como contexto.\nIncluye objetivos, conceptos clave, errores frecuentes, preguntas de práctica y tareas recomendadas."""\n\n@mcp.prompt(title="Generar quiz")\ndef generar_quiz(tema: str, dificultad: str = "media") -> str:\n    """Genera una plantilla para crear un quiz del curso."""\n    return f"""Crea un quiz en español sobre {tema} con dificultad {dificultad}.\nIncluye 5 preguntas de selección múltiple, 2 preguntas cortas y respuestas al final."""\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Servidor MCP del curso")\n    parser.add_argument("--transport", choices=["stdio", "streamable-http"], default=os.environ.get("CURSO_MCP_TRANSPORT", "stdio"))\n    parser.add_argument("--host", default=os.environ.get("CURSO_MCP_HOST", "127.0.0.1"))\n    parser.add_argument("--port", type=int, default=int(os.environ.get("CURSO_MCP_PORT", "8000")))\n    args = parser.parse_args()\n\n    print(f"curso-mcp usando DB: {DB_PATH}", file=sys.stderr)\n    if args.transport == "stdio":\n        mcp.run(transport="stdio")\n    else:\n        print(f"curso-mcp HTTP en http://{args.host}:{args.port}/mcp", file=sys.stderr)\n        mcp.run(transport="streamable-http", host=args.host, port=args.port)\n'

SERVER_PATH.write_text(server_code, encoding="utf-8")
print("Servidor escrito en:", SERVER_PATH)
print(SERVER_PATH.read_text(encoding="utf-8")[:1200])


Servidor escrito en: /content/mcp_lab/servidor_curso_mcp.py

import argparse
import json
import os
import sqlite3
import sys
from pathlib import Path
from typing import Any, Optional

from mcp.server import MCPServer

DB_PATH = Path(os.environ.get("CURSO_MCP_DB", "curso_mcp.db")).resolve()
mcp = MCPServer("curso-mcp")

def conectar():
    if not DB_PATH.exists():
        raise ValueError(f"No existe la base SQLite: {DB_PATH}")
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    return con

def filas_dict(cursor):
    return [dict(row) for row in cursor.fetchall()]

def validar_prioridad(prioridad: str) -> str:
    prioridad = prioridad.lower().strip()
    validas = {"baja", "media", "alta", "critica"}
    if prioridad not in validas:
        raise ValueError(f"Prioridad inválida: {prioridad}. Usa una de: {sorted(validas)}")
    return prioridad

def validar_estado(estado: str) -> str:
    estado = estado.lower().strip()
    validos = {"pendiente", "en_progreso", "c

## Paso 5: compilar servidor

In [100]:
import py_compile
py_compile.compile(str(SERVER_PATH), doraise=True)
print("Servidor compila correctamente.")

Servidor compila correctamente.


## Paso 6: crear cliente MCP por `stdio`

In [107]:
import asyncio
import nest_asyncio
from pydantic import AnyUrl
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(SERVER_PATH)],
    env={
        **os.environ,
        "CURSO_MCP_DB": str(DB_PATH),
    },
)
print(server_params)

command='/usr/bin/python3' args=['/content/mcp_lab/servidor_curso_mcp.py'] env={'SHELL': '/bin/bash', 'COLAB_JUPYTER_TRANSPORT': 'ipc', 'CGROUP_MEMORY_EVENTS': '/sys/fs/cgroup/memory.events /var/colab/cgroup/jupyter-children/memory.events', 'VM_GCE_METADATA_HOST': '169.254.169.253', 'MODEL_PROXY_HOST': 'https://mp.kaggle.net', 'HOSTNAME': '41612eb5de2b', 'LANGUAGE': 'en_US', 'TBE_RUNTIME_ADDR': '172.28.0.1:8011', 'GCE_METADATA_TIMEOUT': '3', 'COLAB_JUPYTER_IP': '172.28.0.12', 'KMP_LISTEN_PORT': '6000', 'TF_FORCE_GPU_ALLOW_GROWTH': 'true', 'ENV': '/root/.bashrc', 'PWD': '/', 'TBE_EPHEM_CREDS_ADDR': '172.28.0.1:8009', 'TBE_CREDS_ADDR': '172.28.0.1:8008', 'COLAB_JUPYTER_TOKEN': '', 'LAST_FORCED_REBUILD': '20260508', 'TCLLIBPATH': '/usr/share/tcltk/tcllib1.20', 'COLAB_KERNEL_MANAGER_PROXY_HOST': '172.28.0.12', 'UV_BUILD_CONSTRAINT': '', 'USE_AUTH_EPHEM': '1', 'COLAB_WARMUP_DEFAULTS': '1', 'HOME': '/root', 'LANG': 'en_US.UTF-8', 'COLAB_ENABLE_SSH': '1', 'CLOUDSDK_CONFIG': '/content/.config'

## Paso 7: listar tools, resources y prompts

In [113]:
async def inspeccionar_servidor():
    with LOG_PATH.open("w", encoding="utf-8") as errlog:
      async with stdio_client(server_params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            resources = await session.list_resources()
            templates = await session.list_resource_templates()
            prompts = await session.list_prompts()
            return {
                    "tools": [
                        {
                            "name": tool.name,
                            "description": tool.description,
                        }
                        for tool in tools.tools
                    ],
                    "resources": [
                        {
                            "uri": str(resource.uri),
                            "name": resource.name,
                            "mime_type": resource.mime_type,
                        }
                        for resource in resources.resources
                    ],
                    "resource_templates": [
                        {
                            "uri_template": template.uri_template,
                            "name": template.name,
                        }
                        for template
                        in templates.resource_templates
                    ],
                    "prompts": [
                        {
                            "name": prompt.name,
                            "description": prompt.description,
                        }
                        for prompt in prompts.prompts
                    ],
                }

info_mcp = await inspeccionar_servidor()
print(json.dumps(info_mcp,ensure_ascii=False,indent=2,))

{
  "tools": [
    {
      "name": "buscar_clases",
      "description": "Busca clases por título, tema o resumen."
    },
    {
      "name": "obtener_clase",
      "description": "Obtiene una clase y sus tareas asociadas."
    },
    {
      "name": "listar_tareas",
      "description": "Lista tareas opcionalmente filtradas por estado y prioridad."
    },
    {
      "name": "crear_tarea",
      "description": "Crea una tarea. Requiere confirmado=True porque escribe en la base."
    },
    {
      "name": "marcar_tarea",
      "description": "Cambia el estado de una tarea. Requiere confirmado=True."
    },
    {
      "name": "resumen_progreso",
      "description": "Devuelve métricas de avance del curso y tareas."
    }
  ],
  "resources": [
    {
      "uri": "curso://resumen",
      "name": "recurso_resumen_curso",
      "mime_type": "application/json"
    },
    {
      "uri": "schema://sqlite",
      "name": "recurso_schema",
      "mime_type": "text/plain"
    }
  ],
  "resourc

## Paso 8: leer resources

Los resources son contexto controlado por la aplicación y no deberían tener efectos secundarios.

In [116]:
async def leer_recursos_demo():
    with LOG_PATH.open("w", encoding="utf-8") as errlog:
      async with stdio_client(server_params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            resumen = await session.read_resource("curso://resumen")
            clase = await session.read_resource("clase://4")
            schema = await session.read_resource("schema://sqlite")
            return resumen, clase, schema

resumen, clase, schema = await leer_recursos_demo()
for nombre, recurso in [("curso://resumen", resumen), ("clase://4", clase), ("schema://sqlite", schema)]:
    print("=" * 80)
    print(nombre)
    contenido = recurso.contents[0]
    print(contenido.text[:1500] if hasattr(contenido, "text") else contenido)

curso://resumen
{
  "clases": [
    {
      "numero": 1,
      "titulo": "Cuantización, PEFT y despliegue",
      "tema": "LLMs eficientes",
      "resumen": "Reduce memoria, adapta modelos con LoRA/QLoRA y despliega de forma responsable.",
      "duracion_min": 120
    },
    {
      "numero": 2,
      "titulo": "Aprendizaje por contexto",
      "tema": "Prompting avanzado",
      "resumen": "Diseña prompts zero-shot, few-shot, CoT, autoconsistencia y salida estructurada.",
      "duracion_min": 120
    },
    {
      "numero": 3,
      "titulo": "Agentes",
      "tema": "Agentes con herramientas",
      "resumen": "Construye agentes que perciben, planifican, actúan y registran memoria.",
      "duracion_min": 120
    },
    {
      "numero": 4,
      "titulo": "Model Context Protocol",
      "tema": "Integración estándar",
      "resumen": "Conecta hosts LLM con tools, resources y prompts mediante MCP.",
      "duracion_min": 120
    }
  ],
  "tareas": [
    {
      "id": 1,
      "c

## Paso 9: llamar tools de lectura

In [119]:
async def llamar_tools_lectura():
    with LOG_PATH.open("w", encoding="utf-8") as errlog:
      async with stdio_client(server_params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            busqueda = await session.call_tool("buscar_clases", {"termino": "agentes"})
            clase = await session.call_tool("obtener_clase", {"numero": 3})
            tareas = await session.call_tool("listar_tareas", {"estado": "pendiente"})
            progreso = await session.call_tool("resumen_progreso", {})
            return busqueda, clase, tareas, progreso

resultados = await llamar_tools_lectura()

for nombre, resultado in zip(["buscar_clases","obtener_clase","listar_tareas","resumen_progreso",],resultados,):
    print("=" * 80)
    print(nombre)
    print("is_error:",getattr(resultado, "is_error", False))

    structured = getattr(resultado,"structured_content",None)

    if structured is not None:
        print(json.dumps(structured,ensure_ascii=False,indent=2)[:2000])
    else:
        for bloque in resultado.content:
            if hasattr(bloque, "text"):
                print(bloque.text[:2000])
            else:
                print(bloque)

buscar_clases
is_error: False
{
  "result": [
    {
      "numero": 3,
      "titulo": "Agentes",
      "tema": "Agentes con herramientas",
      "resumen": "Construye agentes que perciben, planifican, actúan y registran memoria.",
      "duracion_min": 120
    }
  ]
}
obtener_clase
is_error: False
{
  "clase": {
    "numero": 3,
    "titulo": "Agentes",
    "tema": "Agentes con herramientas",
    "resumen": "Construye agentes que perciben, planifican, actúan y registran memoria.",
    "duracion_min": 120
  },
  "tareas": [
    {
      "id": 3,
      "clase_numero": 3,
      "titulo": "Conectar agente a Firestore simulado",
      "prioridad": "alta",
      "estado": "pendiente"
    }
  ]
}
listar_tareas
is_error: False
{
  "result": [
    {
      "id": 1,
      "clase_numero": 1,
      "titulo": "Probar LoRA con modelo pequeño",
      "prioridad": "media",
      "estado": "pendiente"
    },
    {
      "id": 3,
      "clase_numero": 3,
      "titulo": "Conectar agente a Firestore simul

## Paso 10: tools con efectos secundarios y aprobación

`crear_tarea` y `marcar_tarea` modifican la base. El servidor exige `confirmado=True`.

In [121]:
async def probar_sin_confirmar():
    with LOG_PATH.open("w", encoding="utf-8") as errlog:
      async with stdio_client(server_params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await session.call_tool("crear_tarea", {
                "clase_numero": 4,
                "titulo": "Documentar ejecución MCP en Colab/local",
                "prioridad": "alta",
                "confirmado": False,
            })

sin_confirmar = asyncio.get_event_loop().run_until_complete(probar_sin_confirmar())
print("isError:", sin_confirmar.is_error)
for c in sin_confirmar.content:
    if isinstance(c, types.TextContent):
        print(c.text)

isError: True
Error executing tool crear_tarea: Crear tareas modifica la base. Reintenta con confirmado=True si el usuario lo aprueba.


In [123]:
async def crear_y_marcar_confirmada():
    with LOG_PATH.open("w", encoding="utf-8") as errlog:
      async with stdio_client(server_params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            creada = await session.call_tool("crear_tarea", {
                "clase_numero": 4,
                "titulo": "Documentar ejecución MCP en Colab/local",
                "prioridad": "alta",
                "confirmado": True,
            })
            nueva_id = creada.structured_content["id"]
            marcada = await session.call_tool("marcar_tarea", {
                "tarea_id": nueva_id,
                "estado": "en_progreso",
                "confirmado": True,
            })
            progreso = await session.call_tool("resumen_progreso", {})
            return creada, marcada, progreso

creada, marcada, progreso = asyncio.get_event_loop().run_until_complete(crear_y_marcar_confirmada())
print("Tarea creada:")
print(json.dumps(creada.structured_content, ensure_ascii=False, indent=2))
print("\nTarea marcada:")
print(json.dumps(marcada.structured_content, ensure_ascii=False, indent=2))
print("\nProgreso:")
print(json.dumps(progreso.structured_content, ensure_ascii=False, indent=2))


Tarea creada:
{
  "id": 6,
  "clase_numero": 4,
  "titulo": "Documentar ejecución MCP en Colab/local",
  "prioridad": "alta",
  "estado": "pendiente"
}

Tarea marcada:
{
  "id": 6,
  "clase_numero": 4,
  "titulo": "Documentar ejecución MCP en Colab/local",
  "prioridad": "alta",
  "estado": "en_progreso"
}

Progreso:
{
  "total_clases": 4,
  "total_tareas": 6,
  "tareas_por_estado": [
    {
      "estado": "completada",
      "cantidad": 1
    },
    {
      "estado": "en_progreso",
      "cantidad": 1
    },
    {
      "estado": "pendiente",
      "cantidad": 4
    }
  ],
  "tareas_criticas_pendientes": [
    {
      "id": 4,
      "clase_numero": 4,
      "titulo": "Crear servidor MCP por stdio",
      "prioridad": "critica",
      "estado": "pendiente"
    }
  ]
}


## Paso 11: obtener prompts reutilizables

In [124]:
async def obtener_prompts_demo():
    with LOG_PATH.open("w", encoding="utf-8") as errlog:
      async with stdio_client(server_params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            p1 = await session.get_prompt("preparar_estudio", arguments={"numero_clase": "4"})
            p2 = await session.get_prompt("generar_quiz", arguments={"tema": "MCP", "dificultad": "media"})
            return p1, p2

p1, p2 = asyncio.get_event_loop().run_until_complete(obtener_prompts_demo())
for nombre, prompt in [("preparar_estudio", p1), ("generar_quiz", p2)]:
    print("=" * 80)
    print(nombre)
    for msg in prompt.messages:
        print(msg.role, "->", msg.content)

preparar_estudio
user -> type='text' text='Prepara una guía de estudio para la clase 4.\nUsa el recurso clase://4 como contexto.\nIncluye objetivos, conceptos clave, errores frecuentes, preguntas de práctica y tareas recomendadas.' annotations=None meta=None
generar_quiz
user -> type='text' text='Crea un quiz en español sobre MCP con dificultad media.\nIncluye 5 preguntas de selección múltiple, 2 preguntas cortas y respuestas al final.' annotations=None meta=None


## Paso 12: simular flujo host-agente

Un host real decidiría con un LLM. Aquí hacemos una política simple para mostrar el patrón.

In [129]:
async def asistente_simulado(pregunta: str):
    pregunta_l = pregunta.lower()
    with LOG_PATH.open("w", encoding="utf-8") as errlog:
      async with stdio_client(server_params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            if "pendiente" in pregunta_l or "tarea" in pregunta_l:
                resultado = await session.call_tool("listar_tareas", {"estado": "pendiente"})
                datos = (resultado.structured_content or {}).get("result", [])
                lineas = [f"- {d['titulo']} ({d['prioridad']})" for d in datos]
                return f"Encontré {len(datos)} tareas pendientes:\n" + "\n".join(lineas)
            if "mcp" in pregunta_l or "clase 4" in pregunta_l:
                recurso = await session.read_resource("clase://4")
                return "Contexto recuperado de clase://4:\n" + recurso.contents[0].text[:1200]
            resultado = await session.call_tool("resumen_progreso", {})
            return "Resumen de progreso:\n" + json.dumps(resultado.structured_content, ensure_ascii=False, indent=2)

for pregunta in ["¿Qué tareas pendientes tengo?", "Dame contexto de la clase 4 de MCP", "¿Cómo va el progreso del curso?"]:
    print("=" * 80)
    print("Usuario:", pregunta)
    print(await asistente_simulado(pregunta))


Usuario: ¿Qué tareas pendientes tengo?
Encontré 4 tareas pendientes:
- Probar LoRA con modelo pequeño (media)
- Conectar agente a Firestore simulado (alta)
- Crear servidor MCP por stdio (critica)
- Documentar ejecución MCP en Colab/local (alta)
Usuario: Dame contexto de la clase 4 de MCP
Contexto recuperado de clase://4:
{
  "clase": {
    "numero": 4,
    "titulo": "Model Context Protocol",
    "tema": "Integración estándar",
    "resumen": "Conecta hosts LLM con tools, resources y prompts mediante MCP.",
    "duracion_min": 120
  },
  "tareas": [
    {
      "id": 4,
      "clase_numero": 4,
      "titulo": "Crear servidor MCP por stdio",
      "prioridad": "critica",
      "estado": "pendiente"
    },
    {
      "id": 5,
      "clase_numero": 4,
      "titulo": "Documentar ejecución MCP en Colab/local",
      "prioridad": "alta",
      "estado": "pendiente"
    },
    {
      "id": 6,
      "clase_numero": 4,
      "titulo": "Documentar ejecución MCP en Colab/local",
      "priori

## Ejercicios guiados

1. Agrega un resource `tareas://pendientes` y pruébalo por `stdio`.
2. Modifica `crear_tarea` para rechazar títulos de menos de 8 caracteres.
3. Agrega un prompt `planificar_semana`.
4. Diseña una política de aprobación humana para tools que modifican datos.
5. Amplía las capacidades de un host con una LLM.

## Referencias

- MCP: https://modelcontextprotocol.io/docs/getting-started/intro
- Especificación MCP: https://modelcontextprotocol.io/specification
- SDK Python oficial: https://github.com/modelcontextprotocol/python-sdk
- Documentación del SDK Python: https://py.sdk.modelcontextprotocol.io/
- Construir servidores MCP con Python: https://py.sdk.modelcontextprotocol.io/server/
- Clientes MCP con Python: https://py.sdk.modelcontextprotocol.io/client/
- Streamable HTTP en FastMCP: https://py.sdk.modelcontextprotocol.io/server/#streamable-http-transport
